# **Davies Corpus: Acquisition Workflow**
This workflow ingests a locally downloaded Davies corpus (e.g., COHA, COCA) into a RocksDB database. Unlike Google Books ngrams which are downloaded on-the-fly, Davies corpora must be obtained separately and stored locally before running this notebook.

## **Setup**
### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from ngramprep.ngram_filter.lemmatizer import CachedSpacyLemmatizer
from daviesprep.davies_acquire import ingest_davies_corpus
from daviesprep.davies_filter import filter_davies_corpus, load_stopwords, write_whitelist
from ngramprep.utilities.peek import db_head, db_peek, db_peek_prefix
from ngramprep.utilities.count_items import count_db_items

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Configure
Here we set basic parameters: the corpus name, local path to the downloaded corpus files, and database storage paths. We also have the option of filtering by genre during acquisition. Different Davies corpora contain different genre metadata:
- **COHA/COCA**: single-tag genre metadata
  - COHA tag set: `fic`, `nf`, `mag`, `news`
  - COCA tag set: `fic`, `mag`, `news`, `blog`, `web`, `spok`, `acad`, `tv`, `mov`
- **Movies**: Multi-tag genre metadata
  - Tag set: `action`, `adult`, `adventure`, `animation`, `biography`, `comedy`, `crime`, `documentary`, `drama`, `family`, `fantasy`, `film-noir`, `history`, `horror`, `music`, `musical`, `mystery`, `n/a`, `news`, `reality-tv`, `romance`, `sci-fi`, `short`, `sport`, `talk-show`, `thriller`, `war`, `western`

Set `genre_focus=None` to include all genres.

To retain only particular genres, specify the appropriate tags. For example, when working with COHA or COCA, you could specify `genre_focus=['nf']` to keep only nonfiction texts or `genre_focus=['nf', 'mag']` to keep nonfiction and magazine texts. When using the Movies database, you might specify `genre_focus=['drama']` to retain only texts with drama in their tag set or `genre_focus=['drama', 'comedy']` to keep texts with drama _or_ comedy in their tags. 

In [4]:
corpus_name = 'Movies'
genre_focus = None
bin_size = 5
db_path_stub = f'/scratch/edk202/NLP_corpora/{corpus_name}/'

## **Ingest Corpus into Database**

In [5]:
ingest_davies_corpus(
    db_path_stub = db_path_stub,
    genre_focus=genre_focus,
    chunk_on="scene",
    bin_size=bin_size,
    workers=24,
    write_batch_size=500_000,
    compact_after=True,
    combined_bigrams=None
)

Movies CORPUS ACQUISITION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-01-23 16:19:13

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Corpus path:          /scratch/edk202/NLP_corpora/Movies
Text directory:       /scratch/edk202/NLP_corpora/Movies/text
DB path:              /scratch/edk202/NLP_corpora/Movies/Movies
Text files found:     34
Genre focus:          All genres
Key format:           Year-only (training-ready; genre not stored)
Year bin size:        5
Chunking:             scene
Workers:              24
Batch size:           500,000

Processing Files
════════════════════════════════════════════════════════════════════════════════════════════════════


Files Processed: 100%|█████████████████████████████████████████████████████████| 34/34 [00:11<00:00]



Post-Ingestion Compaction
════════════════════════════════════════════════════════════════════════════════════════════════════
Compaction completed in 0:00:06


Processing complete!

Final Summary
════════════════════════════════════════════════════════════════════════════════════════════════════
Files processed:          34/34
Failed files:             0
Total sentences written:  25,517
Database path:            /scratch/edk202/NLP_corpora/Movies/Movies

Documents skipped (metadata missing):
  4476961
  4476974
  4476975
  4476977
  6739040
  6739042
  6829596
  6929891

Genre breakdown:
  action      3,702 sentences
  adult       94 sentences
  adventure   2,842 sentences
  animation   1,281 sentences
  biography   1,278 sentences
  comedy      7,815 sentences
  crime       3,452 sentences
  documentary 2,614 sentences
  drama       11,277 sentences
  family      1,816 sentences
  fantasy     1,451 sentences
  film-noir   386 sentences
  history     850 sentences
  horror      3,424

## **Filter Database**

In [ ]:
stop_set, stop_lang = load_stopwords("en")

filter_options = {
    'stop_set': stop_set,
    'stop_words_language': stop_lang,
    'lemma_gen': CachedSpacyLemmatizer(),
    'whitelist_batch_size': 5_000
}

always_include_tokens = {
    'he', 'she',
    'him', 'her',
    'his', 'hers',
    'himself', 'herself',
    'man', 'woman',
    'men', 'women',
    'boy', 'girl',
    'boys', 'girls',
    'father', 'mother',
    'fathers', 'mothers',
    'son', 'daughter',
    'sons', 'daughters',
    'brother', 'sister',
    'brothers', 'sisters'
}

filter_davies_corpus(
    db_path_stub=db_path_stub,
    genre_focus=genre_focus,
    workers=128,
    batch_size=5_000,
    create_whitelist=True,
    apply_whitelist=True,
    whitelist_size=30_000,
    whitelist_spell_check=True,
    whitelist_year_range=(1950, 2019),
    compact_after=True,
    always_include=always_include_tokens,
    **filter_options
)

Movies CORPUS FILTERING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-01-23 16:19:47

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Source DB:            /scratch/edk202/NLP_corpora/Movies/Movies
Destination DB:       /scratch/edk202/NLP_corpora/Movies/Movies_filtered
Lowercase:            True
Alpha only:           True
ASCII alpha only:     True
Filter short:         True (min_len=3)
Filter stops:         True
Apply lemmas:         True
Workers:              128
Batch size:           5,000

Processing Sentences
════════════════════════════════════════════════════════════════════════════════════════════════════


Batches Processed: 100%|█████████████████████████████████████████████████████████| 6/6 [00:24<00:00]



Building Whitelist
════════════════════════════════════════════════════════════════════════════════════════════════════
Whitelist path:          /scratch/edk202/NLP_corpora/Movies/Movies_whitelist.txt
Top N tokens:            30,000
Year range:              1950-2019
Spell check:             True



Building token frequencies: 100%|████████████████████████████████████████████████| 6/6 [01:32<00:00]



Years present in corpus within range: 14 years
  Range: 1950 to 2015
  Years: [1950, 1955, 1960, 1965, 1970, 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015]

Filtering tokens by year coverage (must appear in all 14 years)...
Tokens before year filter: 38,852
Tokens after year filter:  11,901
Tokens removed:            26,951

Writing whitelist to /scratch/edk202/NLP_corpora/Movies/Movies_whitelist.txt...
Whitelist written successfully: /scratch/edk202/NLP_corpora/Movies/Movies_whitelist.txt

Applying Whitelist
════════════════════════════════════════════════════════════════════════════════════════════════════

Replacing non-whitelist tokens with <UNK>...



Batches Processed: 100%|█████████████████████████████████████████████████████████| 6/6 [00:12<00:00]



Whitelist application complete!
Sentences processed:      25,517
Sentences modified:       1

Post-Filter Compaction
════════════════════════════════════════════════════════════════════════════════════════════════════
Initial DB size:         1.04 GB
Compaction completed in 0:00:03
Size before:             1.04 GB
Size after:              1.04 GB
Space saved:             -198.64 KB (-0.0%)

Processing complete!

Final Summary
════════════════════════════════════════════════════════════════════════════════════════════════════
Sentences read:           25,517
Writes accumulated:       25,517
Sentences rejected:       0
Retention rate:           100.0%
Destination DB:           /scratch/edk202/NLP_corpora/Movies/Movies_filtered

End Time: 2026-01-23 16:22:21
Total Runtime: 0:02:33.881653



## **Optional: Inspect Database Files**

### `db_head`: Show first N records

In [9]:
db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}'

db_head(db, n=5)

First 5 key-value pairs:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   [1930] # Day of days # #Wonderful day of love # # Beauteous day # #Let the sun rise above # #Glorious day # #Making the sun appear # #Beauty reigns # #Over this day of days # # Day of days # # Wonderful day of_love # #Beauteous day # #Let the sun rise above # #Glorious day # #Making the sun appear # #Beauty reigns # #Over this day # #Of days ## Papa Papa Now that 's the third time she 's run away from me It 's disgraceful It 's unbelievable Why do you give me nothing but trouble As soon as you 're born your mother runs away You 're going to be married and your bride runs away It 's nothing but run run run If only it had n't rained What has the rain got to do with it In the sunlight in the moonlight nothing could stop her But other women run away too But after the wedding That 's different That 's all right Why would have waited that long Ah but not s

### `db_peek`: Show records starting from a key

In [23]:
db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}_filtered'

db_peek(db, start_key="[1990] hello", n=5)

5 key-value pairs starting from 000007c668656c6c6f:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   [1990] hello <UNK> <UNK> <UNK> <UNK> <UNK> do <UNK> need <UNK> <UNK> <UNK> <UNK> <UNK> jeweler <UNK> give <UNK> problem he need <UNK> disappear he <UNK> <UNK> <UNK> little <UNK> <UNK> <UNK> <UNK> good <UNK> <UNK> <UNK> know <UNK> <UNK> owe <UNK> <UNK> debt <UNK> <UNK> important <UNK> <UNK> <UNK> want <UNK> see <UNK> old neighborhood <UNK> <UNK> <UNK> call <UNK> <UNK> <UNK> get <UNK> <UNK> <UNK> take <UNK> <UNK> lee <UNK> <UNK> chance come <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> come <UNK> <UNK> get <UNK> <UNK> <UNK> <UNK> whole story arch <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> kill him <UNK> <UNK> long time ago <UNK> day <UNK> look away <UNK> hope he <UNK> change <UNK> <UNK> <UNK> <UNK> tell <UNK> <UNK> <UNK> like <UNK> tell her <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> hold <UNK> door <UNK> huh <UNK> get <UNK> tea <UN

### `db_peek_prefix`: Records matching a prefix

In [42]:
db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}_filtered'

db_peek_prefix(db, prefix="[2000] police", n=5)

5 key-value pairs with prefix 000007d0706f6c696365:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   [2000] police dispatcher <UNK> <UNK> <UNK> confirm <UNK> <UNK> <UNK> vine <UNK> <UNK> possible <UNK> <UNK> <UNK> detail man officer <UNK> scene negative <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> vine <UNK> <UNK> <UNK> <UNK> <UNK> come yes <UNK> <UNK> minute <UNK> <UNK> just <UNK> <UNK> make sure somebody else <UNK> come <UNK> <UNK> <UNK> <UNK> goddamn fender bender <UNK> <UNK> get <UNK> comfortable shit <UNK> <UNK> <UNK> witness <UNK> <UNK> <UNK> carol <UNK> right good <UNK> happen <UNK> carol <UNK> sorry <UNK> <UNK> <UNK> right just get <UNK> turn <UNK> face <UNK> something next time yes <UNK> <UNK> <UNK> just <UNK> little slow right now <UNK> <UNK> <UNK> <UNK> witness <UNK> <UNK> <UNK> <UNK> car park <UNK> <UNK> <UNK> <UNK> <UNK> car <UNK> <UNK> <UNK> park <UNK> good see <UNK> <UNK> <UNK> hit <UNK> <UNK> <UNK> <UNK> accid

## **Optional: Count Database Items**

In [43]:
raw_db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}'

raw_count = count_db_items(raw_db)

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Movies/Movies
Progress interval: every 10,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────



┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ COUNT COMPLETE                                                                                   │
├──────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Items: 25,517                                                                                    │
│ Elapsed: 1.89s                                                                                   │
│ Avg rate: 13,470 items/sec                                                                       │
│ Database: /scratch/edk202/NLP_corpora/Movies/Movies                                              │
└──────────────────────────────────────────────────────────────────────────────────────────────────┘


In [44]:
filtered_db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}_filtered'

filtered_count = count_db_items(filtered_db)

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Movies/Movies_filtered
Progress interval: every 10,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────

┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ COUNT COMPLETE                                                                                   │
├──────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Items: 25,516                                                                                    │
│ Elapsed: 2.13s                                                                                   │
│ Avg rate: 11,953 items/sec                                                                       │
│ Database: /scratch/edk202/NLP_corpora/Movies/Movies_filtered    

In [45]:
filtered_db = f'/scratch/edk202/NLP_corpora/{corpus_name}/{corpus_name}_filtered'

count_per_bin = count_db_items(filtered_db, progress_interval=1_000_000, grouping='year_bin')

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Movies/Movies_filtered
Progress interval: every 1,000,000 items
Grouping by: year_bin

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────

┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ COUNT COMPLETE                                                                                   │
├──────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Items: 25,516                                                                                    │
│ Elapsed: 4.76s                                                                                   │
│ Avg rate: 5,360 items/sec                                                                        │
│ Database: /scratch/edk202/NLP_corpora/Movie